# Guide
https://docs.databricks.com/aws/en/security/secrets/?language=Databricks%C2%A0workspace%C2%A0UI

## Create secret
sh command not working here: databricks secrets create-scope Open-Universe
https://adb-4291509454545364.4.azuredatabricks.net/?o=4291509454545364#secrets/createScope

In [0]:
%sh
databricks secrets create-scope Open-Universe

In [0]:
%sql
def mount_storage(container , environment):
    """ Remount. Inspired by https://github.com/Open-Dataplatform/utils-databricks/blob/main/src/custom_utils/dp_storage/connector.py """
    mount_point = f"/mnt/dp{container}mystorage{environment}"
    print(f"Unmount {mount_point}... ")

    dbutils.fs.unmount(mount_point)

    print('Lookup secrets...')
    tenant_id = dbutils.secrets.get(scope="Open-Universe",key="tenantid")
    client_id = dbutils.secrets.get(scope="Open-Universe",key=f"clientid-databricks-sp-{environment}")
    client_secret =  dbutils.secrets.get(scope="Open-Universe",key=f"pwd-databricks-sp-{environment}")
    account_name = dbutils.secrets.get(scope="my-key-vault",key=f"accountname-storage-{environment}")
    
    configs = {"fs.azure.account.auth.type": "OAuth",
               "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
               "fs.azure.account.oauth2.client.id": client_id,
               "fs.azure.account.oauth2.client.secret": client_secret,
               "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"}
    
    print(f"Mount {mount_point}...")
    dbutils.fs.mount(source=f"abfss://{container}@dp{container}storage{environment}.dfs.core.windows.net/",
                     mount_point=mount_point,
                     extra_configs=configs)

In [0]:
%sql
dbutils.fs.mounts()

In [0]:
%sql
mount_storage(container , environment)